#QLoRA-based fine-tuning

#perform QLoRA-based fine-tuning using a 4-bit quantized model and includes:

BitsAndBytesConfig for memory-efficient model loading.

Proper handling of device maps and fp16 training.

Gemini API compatibility ensured by using a smaller model (falcon-rw-1b).

##Step 0: Install and Import Dependencies

In [2]:
!pip install -U bitsandbytes

In [1]:
# Step 1: Load a synthetic dataset for fine-tuning
from datasets import Dataset

# Create a simple dataset with short general-purpose text prompts and responses
samples = [
    {"text": "What is AI? AI stands for Artificial Intelligence."},
    {"text": "Python is a popular programming language."},
    {"text": "The capital of France is Paris."},
    {"text": "Machine learning is a subset of AI."},
    {"text": "Water freezes at 0 degrees Celsius."}
]

# Convert the list of dictionaries into a Hugging Face Dataset object
dataset = Dataset.from_list(samples)







In [3]:

# Step 2: Load tokenizer and tokenize dataset
from transformers import AutoTokenizer

base_model = "tiiuae/falcon-rw-1b"  # small LLM suitable for quick experimentation

# Load tokenizer for the base model
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Set pad token to eos_token to avoid tokenization errors
tokenizer.pad_token = tokenizer.eos_token

# Define tokenization function for the dataset
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the entire dataset
tokenized_dataset = dataset.map(tokenize, batched=True)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
# Step 3: Load model without quantization (CPU fallback)

# Import PyTorch library for tensor operations and deep learning
import torch  

# Import the causal language model class from Hugging Face Transformers
from transformers import AutoModelForCausalLM  

# Load the pretrained model in float32 precision (CPU-friendly, no quantization)
model = AutoModelForCausalLM.from_pretrained(
    base_model,                 # The model identifier (e.g., "gpt2", "meta-llama/Llama-2-7b")
    torch_dtype=torch.float32,  # Use 32-bit floating point (ensures compatibility with CPU execution)
    device_map=None             # No automatic GPU/TPU mapping → model will default to CPU
)


In [ ]:
# Step 4: Prepare the model for QLoRA fine-tuning

# Import necessary modules from PEFT (Parameter-Efficient Fine-Tuning) library
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType  

# Prepare the base model for k-bit (quantized) training
# This adjusts certain layers (like normalization and embeddings) to be compatible with QLoRA
model = prepare_model_for_kbit_training(model)  

# Define LoRA (Low-Rank Adaptation) configuration
lora_config = LoraConfig(
    r=8,                           # Rank of the low-rank decomposition (controls adapter size)
    lora_alpha=32,                 # Scaling factor for LoRA updates
    target_modules=["query_key_value"],  # Specific layers in the model to apply LoRA (usually attention layers)
    lora_dropout=0.05,             # Dropout rate for LoRA layers to reduce overfitting
    bias="none",                   # Whether to add bias parameters ("none", "all", or "lora_only")
    task_type=TaskType.CAUSAL_LM   # Define the type of task → here it’s Causal Language Modeling
)

# Inject LoRA adapters into the model based on the above configuration
model = get_peft_model(model, lora_config)  

# Print the number of trainable parameters vs total parameters
# (helps confirm that only a small portion of the model is fine-tuned, saving memory & compute)
model.print_trainable_parameters()  


trainable params: 1,572,864 || all params: 1,313,198,080 || trainable%: 0.1198


In [ ]:
# Step 5: Set up training loop using Hugging Face Trainer

# Import required classes for training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling  

# Define training arguments for Hugging Face Trainer
training_args = TrainingArguments(
    output_dir="./outputs",          # Directory where checkpoints and logs will be saved
    num_train_epochs=1,              # Number of training epochs (how many times the dataset is repeated)
    per_device_train_batch_size=2,   # Batch size per device (small value for memory efficiency)
    gradient_accumulation_steps=2,   # Accumulate gradients for 2 steps → effective batch size = 2 x 2 = 4
    logging_steps=1,                 # Log training metrics every step
    learning_rate=2e-4,              # Initial learning rate for optimizer
    fp16=True,                       # Enable mixed precision (fp16) for faster training & lower memory usage
    save_total_limit=1,              # Keep only the most recent checkpoint to save disk space
    report_to="none"                 # Disable reporting to external loggers like WandB or TensorBoard
)

# Define a data collator to dynamically batch and format data
# Here, we set mlm=False because we are training for causal language modeling, not masked LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)  

# Set up the Hugging Face Trainer
trainer = Trainer(
    model=model,                     # The model to fine-tune (with LoRA adapters already injected)
    args=training_args,              # Training arguments defined above
    train_dataset=tokenized_dataset, # Preprocessed/tokenized dataset for training
    data_collator=data_collator      # Function to batch and format data properly
)

# Start fine-tuning the model
trainer.train()  


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
1,2.276500
2,1.719100


TrainOutput(global_step=2, training_loss=1.9978116154670715, metrics={'train_runtime': 80.8652, 'train_samples_per_second': 0.062, 'train_steps_per_second': 0.025, 'total_flos': 4647073873920.0, 'train_loss': 1.9978116154670715, 'epoch': 1.0})

In [7]:

# Step 6: Run inference with the fine-tuned model
prompt = "What is AI?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


What is AI?
Artificial Intelligence (AI) is a branch of computer science that deals with the creation of computer systems that can perform tasks that are normally performed by humans.
AI is a broad term that can be used to describe a variety of different technologies.
